# ⚡ Notebook 03 — Model Training: LightGBM + SynapseML

**Goal:** Train a LightGBM model using Fabric's native SynapseML library. LightGBM typically outperforms scikit-learn models on tabular data.

> **Run time:** ~5 min

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import lightgbm as lgb

# Load Feature Store
df = spark.table('silver_credit_risk_features').toPandas()
print(f'Loaded {len(df)} records')

## Step 1 — Prepare Features (LightGBM handles categoricals natively)

In [ ]:
cat_cols = ['AccountType','Branch','Status','LoanPurpose','EmploymentType','HomeOwnership']
for col in cat_cols:
    df[col] = df[col].astype('category')

feature_cols = [
    'CreditScore','AnnualIncome','LoanAmount','EmploymentYears',
    'DebtToIncomeRatio','NumOpenAccounts','NumDelinquencies',
    'MonthsSinceLastDelinquency','LoanTermMonths',
    'TotalTransactions','TotalLoanAmount','AvgTransactionAmount',
    'NumLoans','MaxTransactionAmount',
    'AccountType','LoanPurpose','EmploymentType','HomeOwnership'
]

X = df[feature_cols].fillna(0)
y = df['IsDefault'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## Step 2 — Train LightGBM with MLflow Autolog

In [ ]:
mlflow.set_experiment('CreditRiskScoring')
mlflow.lightgbm.autolog()  # Automatically logs params, metrics, and model!

with mlflow.start_run(run_name='LightGBM'):
    params = {
        'objective':        'binary',
        'metric':           'auc',
        'learning_rate':    0.05,
        'num_leaves':       63,
        'max_depth':        -1,
        'min_child_samples': 20,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'n_estimators':     500,
        'random_state':     42,
        'verbose':          -1
    }

    model_lgb = lgb.LGBMClassifier(**params)
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )

    y_prob_lgb = model_lgb.predict_proba(X_test)[:, 1]
    auc_lgb    = roc_auc_score(y_test, y_prob_lgb)
    ap_lgb     = average_precision_score(y_test, y_prob_lgb)

    mlflow.log_metric('auc_roc', auc_lgb)
    mlflow.log_metric('avg_precision', ap_lgb)

    print(f'\n✅ LightGBM  |  AUC-ROC: {auc_lgb:.4f}  |  Avg Precision: {ap_lgb:.4f}')
    print(classification_report(y_test, model_lgb.predict(X_test), target_names=['No Default','Default']))

## Step 3 — Feature Importance (Top 10)

In [ ]:
import pandas as pd

fi = pd.DataFrame({
    'Feature':   feature_cols,
    'Importance': model_lgb.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

print('Top 10 Features by Importance (LightGBM):')
print(fi.to_string(index=False))

## Step 4 — Cross-Validation (5-fold)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clone(model_lgb), X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'5-Fold CV AUC-ROC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Individual folds: {[round(s,4) for s in cv_scores]}')